## 1. Environment Setup & PyG Installation

Detect the current PyTorch and CUDA versions provided by the Kaggle environment to dynamically format the wheel URL, then install PyTorch Geometric.

In [ ]:
import torch
import os
import subprocess
import sys

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda', 'Please enable GPU (2 T4 GPUs) in Kaggle settings BEFORE installing.'

print('Torch version:', torch.__version__)
print('CUDA version:', torch.version.cuda)

print('\nInstalling PyTorch Geometric...')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch-geometric'], check=True)

torch_v = torch.__version__.split('+')[0]
cuda_v = 'cu' + torch.version.cuda.replace('.', '')
wheel_url = f'https://data.pyg.org/whl/torch-{torch_v}+{cuda_v}.html'

print(f'\nInstalling PyG dependencies from {wheel_url}...')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyg-lib', 'torch-scatter', 'torch-sparse', '-f', wheel_url], check=True)
print('\nInstallation complete! You can proceed to the next cell.')


## 2. Repository Cloning & Path Configuration

Clone the GitHub repository, set up paths to the Kaggle datasets, and create the necessary directory structure for the processed dataset, splits, and artifacts.

In [ ]:
import os
import torch

REPO_ROOT = '/kaggle/working/antenna-gnn'
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/asparagusD/antenna_gnn.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
import sys
sys.path.insert(0, f'{REPO_ROOT}/src')

RAW_CACHE_DIR = '/kaggle/input/antenna-raw-cache-all-grids'
SEED_MASK_DIR = '/kaggle/input/antenna-seed-masks'
WORKING_DIR = '/kaggle/working'

for N in [25, 35, 45, 55]:
    os.makedirs(f'{WORKING_DIR}/data/processed/{N}x{N}', exist_ok=True)
os.makedirs(f'{WORKING_DIR}/splits', exist_ok=True)
os.makedirs(f'{WORKING_DIR}/artifacts', exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda'
print(f'Paths configured. Device: {device}')


## 3. PyG Graph Construction Logic

Core function to convert a raw antenna patch pattern and its S11 spectrum into a PyTorch Geometric `Data` object. This implements node feature engineering and virtual node connectivity.

In [ ]:
import torch
from torch_geometric.data import Data
import numpy as np

def build_pyg_graph(patch_pattern, s11_db, seed_mask, N):
    # Compute seed centroid
    coords = np.argwhere(seed_mask)
    seed_r, seed_c = coords.mean(axis=0)
    
    # Node features: (N*N + 1) nodes, 5 features each
    node_feats = []
    for i in range(N):
        for j in range(N):
            metal    = float(patch_pattern[i, j])
            x_norm   = j / (N - 1)
            y_norm   = i / (N - 1)
            is_seed  = float(seed_mask[i, j])
            dist_f   = np.sqrt((i - seed_r)**2 + (j - seed_c)**2) / N
            node_feats.append([metal, x_norm, y_norm, is_seed, dist_f])
    
    # Virtual global node (index N*N): all zeros except placeholder (-1 for is_seed)
    node_feats.append([0.0, 0.5, 0.5, -1.0, 0.0])
    node_feats = torch.tensor(node_feats, dtype=torch.float32)
    
    # 4-connectivity edges
    edge_src, edge_dst, edge_attr = [], [], []
    etype_map = {(1,1):0, (1,0):1, (0,1):2, (0,0):3}
    for i in range(N):
        for j in range(N):
            idx = i * N + j
            m_ij = int(patch_pattern[i, j])
            for (di, dj, direction) in [(0,1,0),(0,-1,1),(-1,0,2),(1,0,3)]:
                ni, nj = i+di, j+dj
                if 0 <= ni < N and 0 <= nj < N:
                    nidx = ni * N + nj
                    m_nb = int(patch_pattern[ni, nj])
                    etype = etype_map[(m_ij, m_nb)]
                    edge_src.append(idx); edge_dst.append(nidx)
                    edge_attr.append([etype, direction])
    
    # Virtual node edges (connect to all metal pixels only)
    global_idx = N * N
    for i in range(N):
        for j in range(N):
            if patch_pattern[i, j] == 1:
                idx = i * N + j
                edge_src += [global_idx, idx]
                edge_dst += [idx, global_idx]
                edge_attr += [[4, 4], [4, 4]]  # virtual edge type
    
    edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
    edge_attr  = torch.tensor(edge_attr, dtype=torch.float32)
    
    # Target
    y = torch.tensor(s11_db, dtype=torch.float32).unsqueeze(0)  # (1, 201)
    
    return Data(x=node_feats, edge_index=edge_index, edge_attr=edge_attr, y=y)


## 4. In-Memory Dataset Class

This dataset class avoids individual disk reads during graph construction by loading the entire raw cache for a given grid size into RAM. It builds and saves PyG `Data` objects on demand.

In [ ]:
from torch_geometric.data import Dataset

class AntennaMultiScaleDataset(Dataset):
    def __init__(self, grid_size, seed_mask, s11_mean=None, s11_std=None):
        self.grid_size = grid_size
        self.seed_mask = seed_mask
        cache = torch.load(f'{RAW_CACHE_DIR}/raw_cache_{grid_size}x{grid_size}.pt', weights_only=False)
        self.patterns = cache['patterns']
        self.s11 = cache['s11']
        self.is_functioning = cache['is_functioning']
        self.proc_dir = f'{WORKING_DIR}/data/processed/{grid_size}x{grid_size}'
        self.s11_mean = s11_mean
        self.s11_std = s11_std
        super().__init__(root=None, transform=None, pre_transform=None)
        
    def len(self): 
        return len(self.patterns)
        
    def get(self, idx):
        proc_path = f'{self.proc_dir}/sample_{idx}.pt'
        if os.path.exists(proc_path):
            data = torch.load(proc_path, weights_only=False)
        else:
            data = build_pyg_graph(self.patterns[idx].numpy(), self.s11[idx].numpy(),
                                   self.seed_mask, self.grid_size)
            data.grid_size = self.grid_size
            data.is_functioning = int(self.is_functioning[idx].item())
            data.pixel_size_mm = 32.375 / self.grid_size
            torch.save(data, proc_path)
            
        if self.s11_mean is not None and self.s11_std is not None:
            data.y = (data.y - self.s11_mean) / (self.s11_std + 1e-8)
            
        return data


## 5. Dataset Processing & Caching Loop

Iterate over all grid sizes and process their samples into PyG graphs. This includes a robust checkpointing mechanism via `_DONE.txt` markers. If the Kaggle session times out, re-running this cell skips completed grids and resumes the current grid from where it left off.

In [ ]:
from tqdm.auto import tqdm

for N in [25, 35, 45, 55]:
    done_marker = f'{WORKING_DIR}/data/processed/{N}x{N}_DONE.txt'
    if os.path.exists(done_marker):
        print(f'Grid {N}x{N} already processed. Skipping.')
        continue
        
    print(f'\nProcessing Grid {N}x{N}...')
    seed_mask = np.load(f'{SEED_MASK_DIR}/seed_mask_{N}.npy')
    dataset = AntennaMultiScaleDataset(N, seed_mask)
    
    for i in tqdm(range(len(dataset))):
        _ = dataset.get(i)
        
    with open(done_marker, 'w') as f:
        f.write('DONE\n')
        
    print(f'Grid {N}x{N}: {len(dataset)} graphs built.')


## 6. Stratified Data Splitting

Combine sample indices across all 4 grid sizes and perform a stratified 80/10/10 split based on the combination of `grid_size` and `is_functioning` to ensure a balanced distribution across train, validation, and test sets.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
import json

print('Building stratified splits across all grids...')
all_indices = []
all_strata = []

for N in [25, 35, 45, 55]:
    cache = torch.load(f'{RAW_CACHE_DIR}/raw_cache_{N}x{N}.pt', weights_only=False)
    is_func = cache['is_functioning'].numpy()
    
    for i in range(len(is_func)):
        all_indices.append([N, i])
        all_strata.append(f'{N}_{is_func[i]}')

all_indices = np.array(all_indices)
all_strata = np.array(all_strata)

sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, temp_idx = next(sss1.split(all_indices, all_strata))

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(sss2.split(all_indices[temp_idx], all_strata[temp_idx]))

val_idx = temp_idx[val_idx]
test_idx = temp_idx[test_idx]

splits = {
    'train': all_indices[train_idx].tolist(),
    'val': all_indices[val_idx].tolist(),
    'test': all_indices[test_idx].tolist()
}

with open(f'{WORKING_DIR}/splits/indices.json', 'w') as f:
    json.dump(splits, f)

print(f'Total samples: {len(all_indices)}')
for split_name in ['train', 'val', 'test']:
    print(f'{split_name.capitalize()} size: {len(splits[split_name])}')


## 7. S11 Normalization Statistics

Compute the per-frequency-point mean and standard deviation for the S11 spectra using only the training split to prevent data leakage.

In [ ]:
print('Computing S11 normalization statistics from training split...')

train_s11_list = []
for N, idx in splits['train']:
    cache = torch.load(f'{RAW_CACHE_DIR}/raw_cache_{N}x{N}.pt', weights_only=False)
    train_s11_list.append(cache['s11'][idx])

train_s11 = torch.stack(train_s11_list)
s11_mean = train_s11.mean(dim=0)
s11_std = train_s11.std(dim=0)

np.save(f'{WORKING_DIR}/artifacts/s11_mean.npy', s11_mean.numpy())
np.save(f'{WORKING_DIR}/artifacts/s11_std.npy', s11_std.numpy())

print('Saved normalization statistics to artifacts/.')
print(f'Mean range: [{s11_mean.min().item():.3f}, {s11_mean.max().item():.3f}]')
print(f'Std range: [{s11_std.min().item():.3f}, {s11_std.max().item():.3f}]')


## 8. Verification & DataLoader Test

Apply the computed normalization statistics to the dataset and verify that a PyG `DataLoader` outputs correctly normalized, properly batched graph objects.

In [ ]:
from torch_geometric.loader import DataLoader

s11_mean_t = torch.tensor(np.load(f'{WORKING_DIR}/artifacts/s11_mean.npy'))
s11_std_t = torch.tensor(np.load(f'{WORKING_DIR}/artifacts/s11_std.npy'))

class SubsetDataset(Dataset):
    def __init__(self, split_indices, s11_mean, s11_std):
        self.indices = split_indices
        self.datasets = {}
        for N in [25, 35, 45, 55]:
            seed_mask = np.load(f'{SEED_MASK_DIR}/seed_mask_{N}.npy')
            self.datasets[N] = AntennaMultiScaleDataset(N, seed_mask, s11_mean, s11_std)
        super().__init__(root=None, transform=None, pre_transform=None)
            
    def len(self):
        return len(self.indices)
        
    def get(self, idx):
        N, sample_idx = self.indices[idx]
        return self.datasets[N][sample_idx]

train_dataset = SubsetDataset(splits['train'][:500], s11_mean_t, s11_std_t)  # Test with 500 samples
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

batch = next(iter(train_loader))
print(f'Batch num_graphs: {batch.num_graphs}')
print(f'Batch x shape: {batch.x.shape}')
print(f'Batch edge_index shape: {batch.edge_index.shape}')
print(f'Batch y shape: {batch.y.shape}')

assert not torch.isnan(batch.x).any(), 'NaN in features'
assert not torch.isnan(batch.y).any(), 'NaN in targets'
assert batch.y.shape == (32, 201)

y_mean = batch.y.mean().item()
y_std = batch.y.std().item()
print(f'Batch y mean: {y_mean:.3f} (expect ~0)')
print(f'Batch y std: {y_std:.3f} (expect ~1)')


## 9. Package Artifacts for Transfer

Since `/kaggle/working` does not persist between sessions and downstream chunks (6, 7) need this processed data, zip outputs per grid size SEPARATELY (not one giant archive) to stay within Kaggle output limits.

In [ ]:
import subprocess

for N in [25, 35, 45, 55]:
    cmd = f'zip -r -q /kaggle/working/processed_{N}x{N}.zip /kaggle/working/data/processed/{N}x{N}'
    subprocess.run(cmd, shell=True, check=True)
    print(f'Created processed_{N}x{N}.zip')

cmd = 'zip -r -q /kaggle/working/chunk5_splits_artifacts.zip /kaggle/working/splits /kaggle/working/artifacts'
subprocess.run(cmd, shell=True, check=True)
print('Created chunk5_splits_artifacts.zip')

print('\nDownload all 5 zip files from the Kaggle Output panel. Either')
print('(a) upload to Drive at DATA_ROOT/data/processed/{N}x{N}/, DATA_ROOT/splits/, DATA_ROOT/artifacts/ for downstream Colab chunks, or')
print('(b) re-zip together and upload as a new Kaggle Dataset "antenna-processed-graphs" for downstream Kaggle chunks (6, 7).')
